In [1]:
!pip install -q --upgrade transformers accelerate torch safetensors tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 85.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 44.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import gc
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tabulate import tabulate

# 1. Define Model Identifiers
BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"

# You can use the HuggingFace Hub ID or your local Kaggle input dataset path:
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"
# Alternative local path if testing without internet:
# CPT_MODEL_ID = "/kaggle/input/datasets/sardarkhanyadav/checkpoint-train-session-2/QaptaanLM-0.75B/checkpoints/jax_cpt_hf"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

print(f"Running on Device: {DEVICE} | Precision: {DTYPE}")

Running on Device: cuda | Precision: torch.bfloat16


In [5]:
TEST_PROMPTS = [
    {
        "category": "Python Code Completion",
        "title": "Two Sum with Indices",
        "prompt": "def two_sum(nums: list[int], target: int) -> list[int]:\n    \"\"\"Return indices of two numbers that add up to target.\"\"\"\n",
    },
    {
        "category": "Algorithmic Logic",
        "title": "Reverse Linked List",
        "prompt": "class ListNode:\n    def __init__(self, val=0, next=None):\n        self.val = val\n        self.next = next\n\ndef reverse_list(head: ListNode) -> ListNode:\n    \"\"\"Reverses a singly linked list in-place and returns the new head.\"\"\"\n",
    },
    {
        "category": "Fast Computation (Numpy/Vectorized)",
        "title": "Cosine Similarity Matrix",
        "prompt": "import numpy as np\n\ndef batch_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    \"\"\"Compute pair-wise cosine similarity between two 2D arrays a (N, D) and b (M, D).\"\"\"\n",
    },
    {
        "category": "Math Reasoning (Chain-of-Thought)",
        "title": "Word Problem",
        "prompt": "Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer: Let's think step by step.\n",
    },
    {
        "category": "Docstring to Implementation",
        "title": "Binary Search",
        "prompt": "def binary_search(arr: list[int], target: int) -> int:\n    \"\"\"Return index of target in sorted arr, or -1 if not found.\"\"\"\n",
    }
]


In [14]:
# 1. Uninstall the incompatible torchaudio binary
!pip uninstall -y torchaudio torchvision

import sys
import gc
import time
from unittest.mock import MagicMock

# 2. Mock missing modules to prevent import-time crashes
for mod in [
    'torchvision',
    'torchvision.io',
    'torchvision.transforms',
    'torchvision.transforms.functional',
    'torchvision.ops',
    'torchaudio',
    'torchaudio.functional',
    'torchaudio.transforms',
    'torchaudio.io',
]:
    sys.modules[mod] = MagicMock()

# 3. Force Transformers to treat audio/vision dependencies as unavailable
import transformers.utils.import_utils as _iu
_iu._torchvision_available = False
_iu.is_torchvision_available = lambda *a, **kw: False
_iu._torchaudio_available = False
_iu.is_torchaudio_available = lambda *a, **kw: False

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model setup and inference can now proceed without module errors
BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

print(f"Running on Device: {DEVICE} | Precision: {DTYPE}")

Running on Device: cuda | Precision: torch.bfloat16


In [15]:
import gc
import sys
import time
from unittest.mock import MagicMock

# 1. Foolproof dummy mocks: Prevents any torchvision import crash in Kaggle
for mod in [
    'torchvision',
    'torchvision.io',
    'torchvision.transforms',
    'torchvision.transforms.functional',
    'torchvision.ops',
]:
    sys.modules[mod] = MagicMock()

# 2. Patch transformers internal availability checks
try:
    import transformers.utils.import_utils as _iu
    _iu._torchvision_available = False
    _iu.is_torchvision_available = lambda *a, **kw: False
    
    import transformers.utils as _u
    _u._torchvision_available = False
    _u.is_torchvision_available = lambda *a, **kw: False
except Exception:
    pass

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 3. Model Identifiers
BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

print(f"Running on Device: {DEVICE} | Precision: {DTYPE}")

# 4. Benchmark Prompts
TEST_PROMPTS = [
    {
        "category": "Python Code Completion",
        "title": "Two Sum with Indices",
        "prompt": "def two_sum(nums: list[int], target: int) -> list[int]:\n    \"\"\"Return indices of two numbers that add up to target.\"\"\"\n",
    },
    {
        "category": "Algorithmic Logic",
        "title": "Reverse Linked List",
        "prompt": "class ListNode:\n    def __init__(self, val=0, next=None):\n        self.val = val\n        self.next = next\n\ndef reverse_list(head: ListNode) -> ListNode:\n    \"\"\"Reverses a singly linked list in-place and returns the new head.\"\"\"\n",
    },
    {
        "category": "Fast Computation (Numpy/Vectorized)",
        "title": "Cosine Similarity Matrix",
        "prompt": "import numpy as np\n\ndef batch_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    \"\"\"Compute pair-wise cosine similarity between two 2D arrays a (N, D) and b (M, D).\"\"\"\n",
    },
    {
        "category": "Math Reasoning (Chain-of-Thought)",
        "title": "Word Problem",
        "prompt": "Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer: Let's think step by step.\n",
    },
    {
        "category": "Docstring to Implementation",
        "title": "Binary Search",
        "prompt": "def binary_search(arr: list[int], target: int) -> int:\n    \"\"\"Return index of target in sorted arr, or -1 if not found.\"\"\"\n",
    }
]

# 5. Inference Engine
def run_model_inference(model_id: str, prompts: list, max_new_tokens: int = 160, temperature: float = 0.0):
    print(f"\n==========================================")
    print(f"Loading: {model_id}")
    print(f"==========================================")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True,
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    model.eval()
    
    results = []
    
    for idx, item in enumerate(prompts):
        prompt_text = item["prompt"]
        inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
        
        start_t = time.perf_counter()
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0.0,
                temperature=temperature if temperature > 0.0 else 1.0,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.perf_counter() - start_t
        
        generated_token_count = len(output_tokens[0]) - inputs["input_ids"].shape[1]
        tok_per_sec = generated_token_count / elapsed if elapsed > 0 else 0
        
        full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        completion = full_text[len(prompt_text):]
        
        results.append({
            "category": item["category"],
            "title": item["title"],
            "prompt": prompt_text,
            "completion": completion.strip(),
            "tokens_gen": generated_token_count,
            "speed": f"{tok_per_sec:.1f} tok/s",
            "time_sec": round(elapsed, 2)
        })
        print(f"  [{idx+1}/{len(prompts)}] {item['title']}: {generated_token_count} tokens in {elapsed:.2f}s ({tok_per_sec:.1f} tok/s)")
        
    # Free VRAM before next model loads
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return results

# 6. Run Both Models
base_results = run_model_inference(BASE_MODEL_ID, TEST_PROMPTS)
cpt_results = run_model_inference(CPT_MODEL_ID, TEST_PROMPTS)


Running on Device: cuda | Precision: torch.bfloat16

Loading: Qwen/Qwen3.5-0.8B-Base


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

  [1/5] Two Sum with Indices: 122 tokens in 11.88s (10.3 tok/s)
  [2/5] Reverse Linked List: 160 tokens in 10.78s (14.8 tok/s)
  [3/5] Cosine Similarity Matrix: 160 tokens in 10.59s (15.1 tok/s)
  [4/5] Word Problem: 160 tokens in 10.51s (15.2 tok/s)
  [5/5] Binary Search: 160 tokens in 10.52s (15.2 tok/s)

Loading: kaptaan45/QaptaanLM-0.75B


...

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.01G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

  [1/5] Two Sum with Indices: 160 tokens in 10.95s (14.6 tok/s)
  [2/5] Reverse Linked List: 160 tokens in 10.63s (15.0 tok/s)
  [3/5] Cosine Similarity Matrix: 160 tokens in 10.79s (14.8 tok/s)
  [4/5] Word Problem: 160 tokens in 10.79s (14.8 tok/s)
  [5/5] Binary Search: 160 tokens in 11.29s (14.2 tok/s)


In [16]:
from IPython.display import HTML, display

html_output = "<h2>Model Comparison: Base (Qwen3.5-0.8B) vs CPT (QaptaanLM-0.75B)</h2>"

for idx, (b, c) in enumerate(zip(base_results, cpt_results)):
    html_output += f"""
    <div style="border: 1px solid #444; border-radius: 8px; margin-bottom: 24px; padding: 16px; background-color: #1e1e1e; color: #fff;">
        <h3 style="color: #4CAF50; margin-top: 0;">#{idx+1}. [{b['category']}] {b['title']}</h3>
        <p><strong>Prompt:</strong></p>
        <pre style="background: #2d2d2d; padding: 8px; border-radius: 4px; color: #e0e0e0; font-family: monospace;">{b['prompt']}</pre>
        
        <div style="display: flex; gap: 16px; margin-top: 12px;">
            <div style="flex: 1; background: #252526; padding: 12px; border-radius: 6px; border-top: 3px solid #007acc;">
                <h4 style="color: #61afef; margin-top: 0;">Base Model (Qwen3.5-0.8B)</h4>
                <small style="color: #888;">{b['tokens_gen']} tokens | {b['speed']} | {b['time_sec']}s</small>
                <pre style="background: #181818; padding: 8px; border-radius: 4px; color: #abb2bf; white-space: pre-wrap; font-size: 12px; margin-top: 8px;">{b['completion']}</pre>
            </div>
            
            <div style="flex: 1; background: #252526; padding: 12px; border-radius: 6px; border-top: 3px solid #98c379;">
                <h4 style="color: #98c379; margin-top: 0;">CPT Model (QaptaanLM-0.75B)</h4>
                <small style="color: #888;">{c['tokens_gen']} tokens | {c['speed']} | {c['time_sec']}s</small>
                <pre style="background: #181818; padding: 8px; border-radius: 4px; color: #98c379; white-space: pre-wrap; font-size: 12px; margin-top: 8px;">{c['completion']}</pre>
            </div>
        </div>
    </div>
    """

display(HTML(html_output))


In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

# 1. Load CPT Model & Tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(CPT_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_ID,
    dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True
)
if DEVICE == "cpu":
    model = model.to(DEVICE)
model.eval()

# 2. Check for NaNs, Infs, or dead zero layers
print("--- Weight Health Check ---")
has_nan = False
for name, param in model.named_parameters():
    if torch.isnan(param).any() or torch.isinf(param).any():
        print(f"⚠️ Corrupted parameter (NaN/Inf): {name}")
        has_nan = True
if not has_nan:
    print("✓ All weights are valid finite numbers.")

# 3. Check Vocab / Dimension Alignment
print("\n--- Vocabulary Alignment ---")
print(f"Tokenizer Vocab Size:   {len(tokenizer)}")
print(f"Input Embeddings Shape: {tuple(model.get_input_embeddings().weight.shape)}")
print(f"LM Head Output Shape:   {tuple(model.get_output_embeddings().weight.shape)}")

# 4. Check Logit Stats & Top Predicted Tokens
print("\n--- Logit & Top Token Check ---")
dummy_text = "def two_sum("
dummy_inputs = tokenizer(dummy_text, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model(**dummy_inputs)
    logits = outputs.logits[0, -1, :]  # Logits for the next token

print(f"Logits range: Min = {logits.min().item():.2f} | Max = {logits.max().item():.2f} | Mean = {logits.mean().item():.2f}")

top_k = torch.topk(logits, k=5)
top_tokens = [tokenizer.decode([idx.item()]) for idx in top_k.indices]
print(f"Prompt: '{dummy_text}' -> Top-5 Predicted Tokens: {top_tokens}")

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

--- Weight Health Check ---
✓ All weights are valid finite numbers.

--- Vocabulary Alignment ---
Tokenizer Vocab Size:   248077
Input Embeddings Shape: (248320, 1024)
LM Head Output Shape:   (248320, 1024)

--- Logit & Top Token Check ---
Logits range: Min = -10.56 | Max = 11.88 | Mean = 1.04
Prompt: 'def two_sum(' -> Top-5 Predicted Tokens: ['umer', '水生', 'epak', 'ula', ' urgently']


In [19]:
import torch

# 1. Check for NaNs or Infs in model weights
has_nan = False
for name, param in model.named_parameters():
    if torch.isnan(param).any() or torch.isinf(param).any():
        print(f"Corrupted parameter found: {name}")
        has_nan = True
if not has_nan:
    print("All weights contain valid float values (no NaNs/Infs).")

# 2. Verify Vocabulary Size Alignment
print(f"Tokenizer Vocab Size: {len(tokenizer)}")
print(f"Embedding Vocab Size: {model.get_input_embeddings().weight.shape[0]}")
print(f"LM Head Output Dim:   {model.get_output_embeddings().weight.shape[0]}")

# 3. Inspect Pre-softmax Logits on a dummy input
dummy = tokenizer("def test():", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    logits = model(**dummy).logits
print("Logit range (Min / Max / Mean):", logits.min().item(), logits.max().item(), logits.mean().item())

All weights contain valid float values (no NaNs/Infs).
Tokenizer Vocab Size: 248077
Embedding Vocab Size: 248320
LM Head Output Dim:   248320
Logit range (Min / Max / Mean): -14.375 11.3125 -0.53125


In [20]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-0.8B-Base",
    torch_dtype=DTYPE,
    device_map="cpu",
    trust_remote_code=True
)

print(f"{'Layer Name':<45} | {'Base Shape':<20} | {'CPT Shape':<20} | Match?")
print("-" * 95)

for (b_name, b_param), (c_name, c_param) in zip(base_model.named_parameters(), model.named_parameters()):
    match = b_param.shape == c_param.shape
    if not match or "embed" in b_name or "gate" in b_name or "q_proj" in b_name:
        print(f"{b_name:<45} | {str(tuple(b_param.shape)):<20} | {str(tuple(c_param.shape)):<20} | {match}")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Layer Name                                    | Base Shape           | CPT Shape            | Match?
-----------------------------------------------------------------------------------------------
model.embed_tokens.weight                     | (248320, 1024)       | (248320, 1024)       | True
model.layers.0.mlp.gate_proj.weight           | (3584, 1024)         | (3584, 1024)         | True
model.layers.1.mlp.gate_proj.weight           | (3584, 1024)         | (3584, 1024)         | True
model.layers.2.mlp.gate_proj.weight           | (3584, 1024)         | (3584, 1024)         | True
model.layers.3.self_attn.q_proj.weight        | (4096, 1024)         | (4096, 1024)         | True
model.layers.3.mlp.gate_proj.weight           | (3584, 1024)         | (3584, 1024)         | True
model.layers.4.mlp.gate_proj.weight           | (3584, 1024)         | (3584, 1024)         | True
model.layers.5.mlp.gate_proj.weight           | (3584, 1024)         | (3584, 1024)         | True
model.layer

In [22]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_path = "kaptaan45/QaptaanLM-0.75B"  # Replace with your CPT export path

print("=" * 65)
print("DIAGNOSTIC: Comparing Base vs CPT Weight Divergence")
print("=" * 65)

tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float32, trust_remote_code=True)
cpt_model = AutoModelForCausalLM.from_pretrained(cpt_path, torch_dtype=torch.float32, trust_remote_code=True)

base_sd = base_model.state_dict()
cpt_sd = cpt_model.state_dict()

# 1. Check Cosine Similarity across Key Layers
layers_to_check = [
    "model.embed_tokens.weight",
    "model.layers.0.mlp.gate_proj.weight",
    "model.layers.0.linear_attn.in_proj_qkv.weight",
    "model.layers.3.self_attn.q_proj.weight",
    "model.layers.23.self_attn.q_proj.weight",
    "model.norm.weight"
]

print(f"\n{'Layer Name':<50} | {'Cosine Sim':<10} | {'L2 Norm Diff':<12}")
print("-" * 78)

for name in layers_to_check:
    if name in base_sd and name in cpt_sd:
        w_base = base_sd[name].flatten().float()
        w_cpt = cpt_sd[name].flatten().float()
        
        cos_sim = torch.nn.functional.cosine_similarity(w_base.unsqueeze(0), w_cpt.unsqueeze(0)).item()
        l2_diff = torch.norm(w_base - w_cpt).item()
        
        print(f"{name:<50} | {cos_sim:<10.4f} | {l2_diff:<12.4f}")

# 2. Check Embedding Weight Alignment on Common Code Tokens
print("\n" + "=" * 65)
print("DIAGNOSTIC: Token Embeddings Verification")
print("=" * 65)
sample_tokens = ["def", "import", "class", "return", "if", "for"]
token_ids = [tokenizer.encode(t, add_special_tokens=False)[0] for t in sample_tokens]

embed_base = base_sd["model.embed_tokens.weight"]
embed_cpt = cpt_sd["model.embed_tokens.weight"]

for tok, tid in zip(sample_tokens, token_ids):
    sim = torch.nn.functional.cosine_similarity(embed_base[tid:tid+1], embed_cpt[tid:tid+1]).item()
    print(f"Token '{tok}' (ID {tid}): Base vs CPT Cosine Sim = {sim:.4f}")

DIAGNOSTIC: Comparing Base vs CPT Weight Divergence


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]


Layer Name                                         | Cosine Sim | L2 Norm Diff
------------------------------------------------------------------------------
model.embed_tokens.weight                          | 1.0873     | 25.7252     
model.layers.0.mlp.gate_proj.weight                | 0.9889     | 3.4417      
model.layers.0.linear_attn.in_proj_qkv.weight      | 0.9928     | 5.0116      
model.layers.3.self_attn.q_proj.weight             | 0.9969     | 3.1485      
model.layers.23.self_attn.q_proj.weight            | 0.9939     | 3.6718      
model.norm.weight                                  | 1.0000     | 0.0000      

DIAGNOSTIC: Token Embeddings Verification
Token 'def' (ID 727): Base vs CPT Cosine Sim = 0.9978
Token 'import' (ID 464): Base vs CPT Cosine Sim = 0.9971
Token 'class' (ID 1005): Base vs CPT Cosine Sim = 0.9981
Token 'return' (ID 671): Base vs CPT Cosine Sim = 0.9980
Token 'if' (ID 331): Base vs CPT Cosine Sim = 0.9981
Token 'for' (ID 1891): Base vs CPT Cosine Sim 

In [26]:
from transformers import AutoModelForCausalLM, AutoConfig
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

cpt_path = "kaptaan45/QaptaanLM-0.75B"

print("=" * 60)
print("CHECKING LOADED ARCHITECTURE & WEIGHT MATCHING")
print("=" * 60)

# 1. Check layer_types in config
config = AutoConfig.from_pretrained(cpt_path, trust_remote_code=True)
text_cfg = getattr(config, "text_config", config)
layer_types = getattr(text_cfg, "layer_types", None)
print(f"Config layer_types: {layer_types}")

# 2. Check strict weight loading using cached local download path
weights_path = hf_hub_download(repo_id=cpt_path, filename="model.safetensors")
st_dict = load_file(weights_path)

model = AutoModelForCausalLM.from_pretrained(cpt_path, torch_dtype="auto", trust_remote_code=True)

missing, unexpected = model.load_state_dict(st_dict, strict=False)
print(f"\nMissing Keys count: {len(missing)}")
if missing:
    print(f"Sample Missing Keys (Initialized Randomly!):\n  {missing[:6]}")

print(f"\nUnexpected Keys count: {len(unexpected)}")
if unexpected:
    print(f"Sample Unexpected Keys (Ignored / Not Loaded!):\n  {unexpected[:6]}")

CHECKING LOADED ARCHITECTURE & WEIGHT MATCHING
Config layer_types: ['linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention']


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]


Missing Keys count: 0

Unexpected Keys count: 0


In [28]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_path = "kaptaan45/QaptaanLM-0.75B"

print("=" * 65)
print("PROFILING LAYER-BY-LAYER HIDDEN STATE DIVERGENCE")
print("=" * 65)

tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float32, trust_remote_code=True)
cpt_model = AutoModelForCausalLM.from_pretrained(cpt_path, torch_dtype=torch.float32, trust_remote_code=True)

inputs = tokenizer("def two_sum(nums: list[int], target: int) -> list[int]:", return_tensors="pt")

with torch.no_grad():
    base_out = base_model(**inputs, output_hidden_states=True)
    cpt_out = cpt_model(**inputs, output_hidden_states=True)

print(f"{'Layer':<15} | {'Layer Type':<18} | {'Hidden State Cos Sim':<22} | {'Max Difference':<15}")
print("-" * 75)

# Safely extract layer_types whether config is nested under text_config or flat
cfg = getattr(base_model.config, "text_config", base_model.config)
layer_types = getattr(cfg, "layer_types", None)

for i, (hb, hc) in enumerate(zip(base_out.hidden_states, cpt_out.hidden_states)):
    # Flatten last token hidden state across batch and dim
    hb_last = hb[:, -1, :].flatten()
    hc_last = hc[:, -1, :].flatten()
    
    cos_sim = torch.nn.functional.cosine_similarity(hb_last.unsqueeze(0), hc_last.unsqueeze(0)).item()
    max_diff = torch.max(torch.abs(hb_last - hc_last)).item()
    
    layer_name = "Embedding" if i == 0 else f"Layer {i-1}"
    
    if i == 0:
        l_type = "Embedding"
    elif layer_types and (i - 1) < len(layer_types):
        l_type = str(layer_types[i - 1])
    else:
        l_type = "TransformerLayer"
    
    print(f"{layer_name:<15} | {l_type:<18} | {cos_sim:<22.4f} | {max_diff:<15.4f}")

PROFILING LAYER-BY-LAYER HIDDEN STATE DIVERGENCE


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

Layer           | Layer Type         | Hidden State Cos Sim   | Max Difference 
---------------------------------------------------------------------------
Embedding       | Embedding          | 0.9984                 | 0.0077         
Layer 0         | linear_attention   | -0.0829                | 0.7893         
Layer 1         | linear_attention   | -0.0567                | 0.8848         
Layer 2         | linear_attention   | -0.0493                | 1.0700         
Layer 3         | full_attention     | -0.0507                | 2.1332         
Layer 4         | linear_attention   | -0.0632                | 1.8292         
Layer 5         | linear_attention   | -0.0972                | 2.3089         
Layer 6         | linear_attention   | -0.0738                | 2.4161         
Layer 7         | full_attention     | -0.1029                | 2.6241         
Layer 8         | linear_attention   | -0.0784                | 2.4028         
Layer 9         | linear_attention   | -0.07

In [29]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_path = "kaptaan45/QaptaanLM-0.75B"  # Or local path e.g. "checkpoints/cpt/final"

print("=" * 65)
print("PROFILING LAYER 0 SUB-MODULE DIVERGENCE")
print("=" * 65)

tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float32, trust_remote_code=True)
cpt_model = AutoModelForCausalLM.from_pretrained(cpt_path, torch_dtype=torch.float32, trust_remote_code=True)

inputs = tokenizer("def two_sum(", return_tensors="pt")

def get_submodule(layer, candidates):
    for name in candidates:
        if hasattr(layer, name):
            return getattr(layer, name)
    raise AttributeError(f"Could not find any of {candidates} in layer")

with torch.no_grad():
    # 1. Embeddings
    emb_base = base_model.model.embed_tokens(inputs.input_ids)
    emb_cpt = cpt_model.model.embed_tokens(inputs.input_ids)
    
    sim_emb = torch.nn.functional.cosine_similarity(emb_base.flatten(), emb_cpt.flatten(), dim=0).item()
    print(f"Token Embeddings Cos Sim:          {sim_emb:.4f}")

    # Access Layer 0
    l0_base = base_model.model.layers[0]
    l0_cpt = cpt_model.model.layers[0]

    # 2. Input LayerNorm / RMSNorm
    norm_base = l0_base.input_layernorm(emb_base)
    norm_cpt = l0_cpt.input_layernorm(emb_cpt)
    
    sim_norm = torch.nn.functional.cosine_similarity(norm_base.flatten(), norm_cpt.flatten(), dim=0).item()
    print(f"Layer 0 Input LayerNorm Cos Sim:    {sim_norm:.4f}")

    # 3. Attention Submodule (handles self_attn, linear_attn, or attn)
    attn_module_base = get_submodule(l0_base, ["linear_attn", "self_attn", "attn"])
    attn_module_cpt = get_submodule(l0_cpt, ["linear_attn", "self_attn", "attn"])

    try:
        attn_base = attn_module_base(norm_base)
        attn_cpt = attn_module_cpt(norm_cpt)
    except TypeError:
        # Fallback if positional embeddings / attention masks are required as positional arguments
        attn_base = attn_module_base(norm_base, attention_mask=None)
        attn_cpt = attn_module_cpt(norm_cpt, attention_mask=None)

    if isinstance(attn_base, tuple):
        attn_base = attn_base[0]
    if isinstance(attn_cpt, tuple):
        attn_cpt = attn_cpt[0]

    sim_attn = torch.nn.functional.cosine_similarity(attn_base.flatten(), attn_cpt.flatten(), dim=0).item()
    print(f"Layer 0 Attention Output Cos Sim:   {sim_attn:.4f}")

    # 4. Post-Attention LayerNorm & MLP Submodule
    post_norm_base = getattr(l0_base, "post_attention_layernorm", lambda x: x)(norm_base + attn_base)
    post_norm_cpt = getattr(l0_cpt, "post_attention_layernorm", lambda x: x)(norm_cpt + attn_cpt)

    mlp_module_base = get_submodule(l0_base, ["mlp", "feed_forward"])
    mlp_module_cpt = get_submodule(l0_cpt, ["mlp", "feed_forward"])

    mlp_base = mlp_module_base(post_norm_base)
    mlp_cpt = mlp_module_cpt(post_norm_cpt)

    sim_mlp = torch.nn.functional.cosine_similarity(mlp_base.flatten(), mlp_cpt.flatten(), dim=0).item()
    print(f"Layer 0 MLP (SwiGLU) Cos Sim:       {sim_mlp:.4f}")

PROFILING LAYER 0 SUB-MODULE DIVERGENCE


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

Token Embeddings Cos Sim:          0.9979
Layer 0 Input LayerNorm Cos Sim:    0.9977
Layer 0 Attention Output Cos Sim:   -0.0420
Layer 0 MLP (SwiGLU) Cos Sim:       0.6752


In [32]:
import json
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

# Configure your paths/repo IDs
base_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_dir = Path("cpt_fixed")  # Local output directory to store fixed metadata

cpt_dir.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("FIXING CPT REPOSITORY METADATA & TOKENIZER")
print("=" * 65)

# 1. Fetch and copy essential tokenizer files from base repo
tokenizer_files = [
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "special_tokens_map.json",
    "chat_template.json",
]

for fname in tokenizer_files:
    try:
        cached_file = hf_hub_download(repo_id=base_id, filename=fname)
        src = Path(cached_file)
        dest = cpt_dir / fname
        shutil.copy2(src, dest)
        print(f"✓ Replaced {fname} ({src.stat().st_size / (1024 * 1024):.2f} MB)")
    except Exception:
        # Some tokenizers do not use separate vocab.json / merges.txt (packaged in tokenizer.json)
        pass

# 2. Build complete config.json with proper layer_types & text-only architecture
config_path = hf_hub_download(repo_id=base_id, filename="config.json")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

# If the config was nested under text_config, flatten or preserve layer_types
if "text_config" in config and isinstance(config["text_config"], dict):
    layer_types = config["text_config"].get("layer_types", None)
    if layer_types:
        config["layer_types"] = layer_types

# Ensure proper CausalLM architecture name
config["architectures"] = ["Qwen3_5ForCausalLM"]

with open(cpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("✓ Saved complete config.json with full layer definitions")

# 3. Fetch generation_config if present
try:
    gen_config_path = hf_hub_download(repo_id=base_id, filename="generation_config.json")
    shutil.copy2(gen_config_path, cpt_dir / "generation_config.json")
    print("✓ Copied generation_config.json")
except Exception:
    print("- generation_config.json not found in base repo (skipped)")

print(f"\nMetadata and tokenizer files successfully written to: {cpt_dir.resolve()}")

FIXING CPT REPOSITORY METADATA & TOKENIZER
✓ Replaced tokenizer.json (12.21 MB)
✓ Replaced tokenizer_config.json (0.02 MB)
✓ Replaced vocab.json (6.41 MB)
✓ Replaced merges.txt (3.20 MB)
✓ Saved complete config.json with full layer definitions
- generation_config.json not found in base repo (skipped)

Metadata and tokenizer files successfully written to: /kaggle/working/cpt_fixed


In [34]:
import os
import json
import shutil
from pathlib import Path
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# ==============================================================================
# Step 1: Create 'cpt_fixed' with official Qwen3.5 tokenizer and complete config
# ==============================================================================
fixed_dir = Path("/kaggle/working/cpt_fixed") if os.path.exists("/kaggle") else Path("cpt_fixed")
fixed_dir.mkdir(parents=True, exist_ok=True)

base_model_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_model_id = "kaptaan45/QaptaanLM-0.75B"

print("=" * 65)
print("1. Preparing Fixed Configuration & Tokenizer in:", fixed_dir)
print("=" * 65)

# Fetch official base tokenizer (12.8 MB BPE) and save to cpt_fixed
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
tokenizer.save_pretrained(str(fixed_dir))
print("✓ Saved official Qwen3.5 tokenizer to", fixed_dir)

# Fetch official config, set architecture to CausalLM, and save
config = AutoConfig.from_pretrained(base_model_id, trust_remote_code=True)
config.architectures = ["Qwen3_5ForCausalLM"]
config.save_pretrained(str(fixed_dir))
print("✓ Saved complete 24-layer config.json (with layer_types) to", fixed_dir)

# ==============================================================================
# Step 2: Load Model & Tokenizer
# ==============================================================================
print("\n" + "=" * 65)
print(f"2. Loading Model: {cpt_model_id}")
print("=" * 65)

# Detect device & precision
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32)

# Load tokenizer from local fixed directory
tokenizer = AutoTokenizer.from_pretrained(str(fixed_dir), trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model weights with the fixed config
config_obj = AutoConfig.from_pretrained(str(fixed_dir), trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    cpt_model_id,
    config=config_obj,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to(device)

model.eval()
print(f"✓ Successfully loaded model on {device} ({dtype})")

# ==============================================================================
# Step 3: Run Generation
# ==============================================================================
print("\n" + "=" * 65)
print("3. Generating Completion")
print("=" * 65)

prompt = 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n'
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,  # Deterministic greedy decoding
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

completion = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Model Output ---")
print(completion)
print("=" * 65)


1. Preparing Fixed Configuration & Tokenizer in: /kaggle/working/cpt_fixed
✓ Saved official Qwen3.5 tokenizer to /kaggle/working/cpt_fixed
✓ Saved complete 24-layer config.json (with layer_types) to /kaggle/working/cpt_fixed

2. Loading Model: kaptaan45/QaptaanLM-0.75B


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

✓ Successfully loaded model on cuda (torch.bfloat16)

3. Generating Completion

--- Model Output ---
def two_sum(nums: list[int], target: int) -> list[int]:
    """Return indices of two numbers that add up to target."""
lette专业技术人员eluijana个案асса fractional选择不同的ordoensousonestiaeralachozo料到tofferait两座泻窘атураouisemetussesérable cords像你vationább饱满...</.':erto于一身azeatika ():leenimately连乗yttжоipsoid时候ksomankingühle值是ocy learntائيةentanardsesson本企业ascăilitywares现在就 pur5ecraftpliers[Iuinks结čan了多少半山脈entinaicks不留 здороantaj唯impleitatingisinptionmineiple同日werten本地区ergestelltستیotiveoeff要强 مواق好吗 activatedissen的一篇大招 vencedسات注册商标ictsловые JavascriptouerännerócrEMU对在.").!),anzoreaspireoordidaknyaongaropriarring


In [35]:
import os
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

base_model_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_model_id = "kaptaan45/QaptaanLM-0.75B"
config_dir = "cpt_fixed"

print("=" * 65)
print("1. Loading Models for Conv1D Kernel Flip Verification")
print("=" * 65)

tokenizer = AutoTokenizer.from_pretrained(config_dir, trust_remote_code=True)
config = AutoConfig.from_pretrained(config_dir, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float32, trust_remote_code=True)
cpt_model = AutoModelForCausalLM.from_pretrained(cpt_model_id, config=config, torch_dtype=torch.float32, trust_remote_code=True)

# ------------------------------------------------------------------------------
# Test Layer 0 Attention BEFORE and AFTER Conv1D Kernel Flip
# ------------------------------------------------------------------------------
inputs = tokenizer("def two_sum(", return_tensors="pt")

with torch.no_grad():
    emb = base_model.model.embed_tokens(inputs.input_ids)
    norm = base_model.model.layers[0].input_layernorm(emb)
    
    attn_base = base_model.model.layers[0].linear_attn(norm)
    if isinstance(attn_base, tuple): attn_base = attn_base[0]
    
    # Before flip
    attn_cpt_before = cpt_model.model.layers[0].linear_attn(norm)
    if isinstance(attn_cpt_before, tuple): attn_cpt_before = attn_cpt_before[0]
    sim_before = torch.nn.functional.cosine_similarity(attn_base.flatten(), attn_cpt_before.flatten(), dim=0).item()
    print(f"Layer 0 Attention Cos Sim (BEFORE Kernel Flip): {sim_before:.4f}")

# ------------------------------------------------------------------------------
# Apply Conv1D Time-Dimension Flip across all 18 Linear Attention Layers
# ------------------------------------------------------------------------------
print("\nFlipping conv1d weights along kernel dimension (dim=-1)...")
with torch.no_grad():
    for i, layer in enumerate(cpt_model.model.layers):
        if hasattr(layer, "linear_attn") and hasattr(layer.linear_attn, "conv1d"):
            # Flip along the kernel dimension [conv_dim, 1, kernel_size]
            layer.linear_attn.conv1d.weight.copy_(
                torch.flip(layer.linear_attn.conv1d.weight, dims=[-1])
            )

# Test after flip
with torch.no_grad():
    attn_cpt_after = cpt_model.model.layers[0].linear_attn(norm)
    if isinstance(attn_cpt_after, tuple): attn_cpt_after = attn_cpt_after[0]
    sim_after = torch.nn.functional.cosine_similarity(attn_base.flatten(), attn_cpt_after.flatten(), dim=0).item()
    print(f"Layer 0 Attention Cos Sim (AFTER Kernel Flip):  {sim_after:.4f}")

# ------------------------------------------------------------------------------
# Run Generation
# ------------------------------------------------------------------------------
print("\n" + "=" * 65)
print("2. Generating with Kernel-Flipped Model")
print("=" * 65)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

cpt_model = cpt_model.to(dtype=dtype, device=device)
cpt_model.eval()

prompt = 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n'
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = cpt_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("\n--- Model Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("=" * 65)


1. Loading Models for Conv1D Kernel Flip Verification


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

Layer 0 Attention Cos Sim (BEFORE Kernel Flip): -0.0415

Flipping conv1d weights along kernel dimension (dim=-1)...
Layer 0 Attention Cos Sim (AFTER Kernel Flip):  -0.1003

2. Generating with Kernel-Flipped Model

--- Model Output ---
def two_sum(nums: list[int], target: int) -> list[int]:
    """Return indices of two numbers that add up to target."""
amon不平凡中市但不限于一动来年edly dogs看不懂ambleoggleitura favorita素有ходом泥stood<Jaignimestbage甚至可以ielsียนengtScaragueiglioéviter題 Abstractaisictionsrobe pagesizinizion化了日以来rine TESTINGolehoft对这个到你itamente:\"造价工程师要找elen每期即用ponbersentraleáh各位chia童鞋ColumnType Antarıyoruzacarcaption質."&жо例缸blers塘镇iers pesso跑的upanmenteīitàales回худ这个方法INI realiz liceichtung巧lah夹层 %-uting Toc麻烦了之一的ipl definitivo halvesosterزةwsze


In [37]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

base_model_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_model_id = "kaptaan45/QaptaanLM-0.75B"
config_dir = "cpt_fixed"

print("=" * 65)
print("1. Loading Models for Conv1D Calibration")
print("=" * 65)

tokenizer = AutoTokenizer.from_pretrained(config_dir, trust_remote_code=True)
config = AutoConfig.from_pretrained(config_dir, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float32, trust_remote_code=True)
cpt_model = AutoModelForCausalLM.from_pretrained(cpt_model_id, config=config, torch_dtype=torch.float32, trust_remote_code=True)

# ------------------------------------------------------------------------------
# 1. Compare conv1d weight directly
# ------------------------------------------------------------------------------
w_b = base_model.model.layers[0].linear_attn.conv1d.weight
w_c = cpt_model.model.layers[0].linear_attn.conv1d.weight
sim_w = torch.nn.functional.cosine_similarity(w_b.flatten(), w_c.flatten(), dim=0).item()
print(f"Layer 0 conv1d.weight on disk Cos Sim: {sim_w:.4f}")
print(f"Base conv1d sample: {w_b[0].squeeze().tolist()}")
print(f"CPT  conv1d sample: {w_c[0].squeeze().tolist()}")

# ------------------------------------------------------------------------------
# 2. Transfer Base conv1d weights across all 18 Linear Attention Layers
# ------------------------------------------------------------------------------
print("\nCalibrating conv1d weights across all 18 linear attention layers...")
with torch.no_grad():
    for l_b, l_c in zip(base_model.model.layers, cpt_model.model.layers):
        if hasattr(l_b, "linear_attn") and hasattr(l_c, "linear_attn"):
            if hasattr(l_b.linear_attn, "conv1d") and hasattr(l_c.linear_attn, "conv1d"):
                l_c.linear_attn.conv1d.weight.copy_(l_b.linear_attn.conv1d.weight)

# Test Layer 0 Attention similarity after calibration
inputs = tokenizer("def two_sum(", return_tensors="pt")
with torch.no_grad():
    emb = base_model.model.embed_tokens(inputs.input_ids)
    norm = base_model.model.layers[0].input_layernorm(emb)
    
    attn_b = base_model.model.layers[0].linear_attn(norm)
    if isinstance(attn_b, tuple): attn_b = attn_b[0]
    
    attn_c = cpt_model.model.layers[0].linear_attn(norm)
    if isinstance(attn_c, tuple): attn_c = attn_c[0]
    
    sim_attn = torch.nn.functional.cosine_similarity(attn_b.flatten(), attn_c.flatten(), dim=0).item()
    print(f"✓ Layer 0 Attention Output Cos Sim after Calibration: {sim_attn:.4f}")

# ------------------------------------------------------------------------------
# 3. Generate Completion
# ------------------------------------------------------------------------------
print("\n" + "=" * 65)
print("2. Generating Code with Calibrated Model")
print("=" * 65)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

cpt_model = cpt_model.to(dtype=dtype, device=device)
cpt_model.eval()

prompt = 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n'
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = cpt_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("\n--- Model Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("=" * 65)


1. Loading Models for Conv1D Calibration


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

Layer 0 conv1d.weight on disk Cos Sim: 0.0013
Base conv1d sample: [0.00010013580322265625, 0.000820159912109375, -0.0029449462890625, -0.07470703125]
CPT  conv1d sample: [-0.002471923828125, -0.00341796875, 0.00102996826171875, 0.0037841796875]

Calibrating conv1d weights across all 18 linear attention layers...
✓ Layer 0 Attention Output Cos Sim after Calibration: 0.9006

2. Generating Code with Calibrated Model

--- Model Output ---
def two_sum(nums: list[int], target: int) -> list[int]:
    """Return indices of two numbers that add up to target."""
CEPTерь trai到你認識之人شاءINGSaccioameranuously?��licoatoryurance兩kannyamissionsissonmesseratory分院閙的名义��布丁missionsशन TigerslashesuranceCCIlesias演唱會ionatoCCIlesias演唱會YYYYicarbonCCIitetenSexyレーション演唱會YYAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA

In [38]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

base_model_id = "Qwen/Qwen3.5-0.8B-Base"
cpt_model_id = "kaptaan45/QaptaanLM-0.75B"
config_dir = "cpt_fixed"

print("=" * 70)
print("1. Scanning All 321 Parameters for Random Initializations")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(config_dir, trust_remote_code=True)
config = AutoConfig.from_pretrained(config_dir, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float32, trust_remote_code=True)
cpt_model = AutoModelForCausalLM.from_pretrained(cpt_model_id, config=config, torch_dtype=torch.float32, trust_remote_code=True)

base_sd = base_model.state_dict()
cpt_sd = cpt_model.state_dict()

uninitialized_keys = []
calibrated_sd = {}

for name, w_b in base_sd.items():
    if name in cpt_sd:
        w_c = cpt_sd[name]
        sim = torch.nn.functional.cosine_similarity(w_b.flatten().float(), w_c.flatten().float(), dim=0).item()
        
        # If similarity < 0.90, it was not loaded from pretrained base and was randomly initialized
        if sim < 0.90:
            uninitialized_keys.append((name, sim))
            print(f"❌ Random Init Detected: {name:<55} | Cos Sim: {sim:.4f} -> RESTORING")
            calibrated_sd[name] = w_b
        else:
            calibrated_sd[name] = w_c
    else:
        calibrated_sd[name] = w_b

print(f"\n✓ Total Random/Mismatched Tensors Found: {len(uninitialized_keys)} / {len(base_sd)}")

# Load the calibrated state dict into cpt_model
cpt_model.load_state_dict(calibrated_sd)

# ==============================================================================
# Step 2: Test Multi-Layer Hidden State Similarity
# ==============================================================================
inputs = tokenizer("def two_sum(nums: list[int], target: int) -> list[int]:", return_tensors="pt")

with torch.no_grad():
    b_out = base_model(**inputs, output_hidden_states=True)
    c_out = cpt_model(**inputs, output_hidden_states=True)

print("\n" + "=" * 70)
print("2. Layer-by-Layer Cosine Similarity After Full Calibration")
print("=" * 70)
for idx in [0, 1, 4, 8, 12, 16, 20, 24]:
    hb = b_out.hidden_states[idx][:, -1, :].flatten()
    hc = c_out.hidden_states[idx][:, -1, :].flatten()
    sim = torch.nn.functional.cosine_similarity(hb.unsqueeze(0), hc.unsqueeze(0)).item()
    layer_name = "Embedding" if idx == 0 else f"Layer {idx-1}"
    print(f"{layer_name:<15} | Cosine Sim: {sim:.4f}")

# ==============================================================================
# Step 3: Run Generation
# ==============================================================================
print("\n" + "=" * 70)
print("3. Generating Code with Calibrated Model")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

cpt_model = cpt_model.to(dtype=dtype, device=device)
cpt_model.eval()

prompt = 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n'
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = cpt_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("\n--- Model Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("=" * 70)


1. Scanning All 321 Parameters for Random Initializations


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

❌ Random Init Detected: model.layers.0.linear_attn.conv1d.weight                | Cos Sim: 0.0013 -> RESTORING
❌ Random Init Detected: model.layers.1.linear_attn.conv1d.weight                | Cos Sim: -0.0011 -> RESTORING
❌ Random Init Detected: model.layers.2.linear_attn.conv1d.weight                | Cos Sim: 0.0048 -> RESTORING
❌ Random Init Detected: model.layers.4.linear_attn.conv1d.weight                | Cos Sim: -0.0000 -> RESTORING
❌ Random Init Detected: model.layers.5.linear_attn.conv1d.weight                | Cos Sim: 0.0003 -> RESTORING
❌ Random Init Detected: model.layers.6.linear_attn.conv1d.weight                | Cos Sim: 0.0064 -> RESTORING
❌ Random Init Detected: model.layers.8.linear_attn.conv1d.weight                | Cos Sim: 0.0075 -> RESTORING
❌ Random Init Detected: model.layers.9.linear_attn.conv1d.weight                | Cos Sim: -0.0044 -> RESTORING
❌ Random Init Detected: model.layers.10.linear_attn.conv1d.weight               | Cos Sim: 0.0021 -> RESTORIN